# Non-Parametric Tests in Statistics: A Comprehensive Guide

---

## 1. Introduction

Non-parametric tests (also called **distribution-free tests**) are statistical methods that do **not** assume the data follows a specific probability distribution (e.g., normal/Gaussian). Unlike parametric tests (t-tests, ANOVA, Pearson correlation), which require assumptions about population parameters (mean, variance, normality), non-parametric tests make minimal assumptions about the underlying data-generating process.

### Why "Non-Parametric"?

The term arises because these methods do not estimate population **parameters** (like $$\mu$$ or $$\sigma^2$$). Instead, they work with:
- **Ranks** of observations
- **Signs** of differences
- **Frequencies/counts** in categories
- **Order statistics**

---

## 2. When to Use Non-Parametric Tests

| Condition | Use Non-Parametric? |
|-----------|--------------------|
| Data is ordinal (ranked) | Yes |
| Small sample size ($$n < 30$$) with non-normal data | Yes |
| Presence of significant outliers | Yes |
| Skewed distributions | Yes |
| Data violates homogeneity of variance | Yes |
| Median is a better measure of central tendency than mean | Yes |
| Data is nominal/categorical | Yes |

---

## 3. Parametric vs. Non-Parametric: A Comparison

| Aspect | Parametric | Non-Parametric |
|--------|-----------|----------------|
| Distribution assumption | Normal distribution required | No distribution assumed |
| Central tendency | Mean ($$\bar{x}$$) | Median ($$\tilde{x}$$) |
| Data type | Interval/Ratio | Ordinal/Nominal/Interval |
| Statistical power | Higher (when assumptions met) | Lower (trades power for robustness) |
| Sample size requirement | Moderate to large | Works with small samples |
| Sensitivity to outliers | High | Low (rank-based) |
| Computational basis | Distributional parameters | Ranks, signs, permutations |

---

## 4. Advantages and Disadvantages

### Advantages
- Robust to outliers and non-normality
- Applicable to ordinal and nominal data
- Valid for small samples
- Fewer assumptions to verify
- Often simpler to understand conceptually

### Disadvantages
- Lower statistical power when parametric assumptions ARE met (approximately 95% efficiency of parametric counterpart for large samples)
- Cannot easily handle complex interactions
- Confidence intervals are harder to construct
- Less familiar to many practitioners
- May discard information by converting to ranks

# 5. Mann-Whitney U Test (Wilcoxon Rank-Sum Test)

---

## Overview
The Mann-Whitney U test is the non-parametric equivalent of the **independent samples t-test**. It tests whether two independent groups come from the same distribution, specifically whether one group tends to have larger (or smaller) values than the other.

## Hypotheses

$$H_0: P(X > Y) = 0.5$$

(The probability that a randomly selected observation from group 1 exceeds a randomly selected observation from group 2 is 0.5 — i.e., the distributions are identical.)

$$H_1: P(X > Y) \neq 0.5$$

## Assumptions
1. The two samples are **independent**
2. The observations are **ordinal** (at minimum)
3. Under $$H_0$$, the distributions have the **same shape** (differ only in location)
4. All observations are independent of each other

## Mathematical Formulation

### Step 1: Combine and Rank
Combine all $$n_1 + n_2$$ observations and assign ranks $$1, 2, \ldots, N$$ where $$N = n_1 + n_2$$.

### Step 2: Compute Rank Sums
$$R_1 = \sum_{i=1}^{n_1} \text{rank}(x_i), \quad R_2 = \sum_{j=1}^{n_2} \text{rank}(y_j)$$

### Step 3: Compute U Statistics
$$U_1 = n_1 n_2 + \frac{n_1(n_1 + 1)}{2} - R_1$$

$$U_2 = n_1 n_2 + \frac{n_2(n_2 + 1)}{2} - R_2$$

Note: $$U_1 + U_2 = n_1 \cdot n_2$$

The test statistic is: $$U = \min(U_1, U_2)$$

### Step 4: Large-Sample Approximation ($$n_1, n_2 > 20$$)

Under $$H_0$$:

$$\mu_U = \frac{n_1 n_2}{2}$$

$$\sigma_U = \sqrt{\frac{n_1 n_2 (n_1 + n_2 + 1)}{12}}$$

$$z = \frac{U - \mu_U}{\sigma_U}$$

With **tied ranks correction**:

$$\sigma_U = \sqrt{\frac{n_1 n_2}{12} \left[ (N + 1) - \sum_{k=1}^{g} \frac{t_k^3 - t_k}{N(N-1)} \right]}$$

where $$g$$ is the number of tied groups and $$t_k$$ is the size of the $$k$$-th tied group.

## Effect Size: Rank-Biserial Correlation

$$r_{rb} = 1 - \frac{2U}{n_1 n_2}$$

Interpretation: $$|r_{rb}| < 0.3$$ = small, $$0.3 - 0.5$$ = medium, $$> 0.5$$ = large

## Industrial Applications
- **Pharma/Clinical Trials**: Comparing drug efficacy between treatment and control groups when response is ordinal (e.g., pain scale)
- **A/B Testing**: Comparing user engagement metrics (session duration, clicks) between two variants when data is highly skewed
- **Manufacturing**: Comparing defect counts between two production lines
- **HR Analytics**: Comparing satisfaction scores between departments
- **E-commerce**: Comparing purchase amounts between customer segments

In [0]:
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

# =============================================================================
# MANN-WHITNEY U TEST - Complete Working Example
# =============================================================================

# Scenario: Comparing customer satisfaction scores (1-10) between two stores
# Store A uses a new service model, Store B uses the traditional model
store_a = np.array([7, 8, 9, 6, 8, 9, 7, 8, 10, 9, 8, 7, 9, 8, 7])
store_b = np.array([5, 6, 7, 4, 6, 5, 7, 6, 5, 4, 6, 7, 5])

print("=" * 70)
print("MANN-WHITNEY U TEST")
print("=" * 70)
print(f"\nStore A (new model): n={len(store_a)}, median={np.median(store_a)}, mean={store_a.mean():.2f}")
print(f"Store B (traditional): n={len(store_b)}, median={np.median(store_b)}, mean={store_b.mean():.2f}")

# Perform the test
u_stat, p_value = stats.mannwhitneyu(store_a, store_b, alternative='two-sided')

print(f"\n--- Results ---")
print(f"U statistic: {u_stat}")
print(f"P-value: {p_value:.6f}")

# Effect size: Rank-biserial correlation
n1, n2 = len(store_a), len(store_b)
r_rb = 1 - (2 * u_stat) / (n1 * n2)
print(f"Rank-biserial correlation (effect size): {r_rb:.4f}")

# Manual computation for understanding
print(f"\n--- Manual Computation ---")
combined = np.concatenate([store_a, store_b])
labels = np.array(['A'] * n1 + ['B'] * n2)
ranks = stats.rankdata(combined)  # Handles ties with average ranks

R1 = ranks[:n1].sum()
R2 = ranks[n1:].sum()
print(f"Sum of ranks (Store A): R1 = {R1}")
print(f"Sum of ranks (Store B): R2 = {R2}")
print(f"Check: R1 + R2 = {R1 + R2} should equal N(N+1)/2 = {(n1+n2)*(n1+n2+1)/2}")

U1 = n1 * n2 + n1 * (n1 + 1) / 2 - R1
U2 = n1 * n2 + n2 * (n2 + 1) / 2 - R2
print(f"U1 = {U1}, U2 = {U2}")
print(f"U = min(U1, U2) = {min(U1, U2)}")

# Large-sample Z approximation
mu_U = n1 * n2 / 2
sigma_U = np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
z = (min(U1, U2) - mu_U) / sigma_U
print(f"\nZ-approximation: z = {z:.4f}")
print(f"P-value (z-approx): {2 * stats.norm.cdf(z):.6f}")

# Interpretation
alpha = 0.05
print(f"\n--- Interpretation (α = {alpha}) ---")
if p_value < alpha:
    print(f"✓ REJECT H₀: Significant difference between stores (p={p_value:.4f} < {alpha})")
    print(f"  Store A customers report significantly {'higher' if np.median(store_a) > np.median(store_b) else 'lower'} satisfaction.")
else:
    print(f"✗ FAIL TO REJECT H₀: No significant difference (p={p_value:.4f} ≥ {alpha})")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot([store_a, store_b], labels=['Store A (New)', 'Store B (Traditional)'])
axes[0].set_ylabel('Satisfaction Score')
axes[0].set_title('Distribution Comparison')

axes[1].hist(store_a, alpha=0.6, label=f'Store A (Mdn={np.median(store_a)})', bins=6, edgecolor='black')
axes[1].hist(store_b, alpha=0.6, label=f'Store B (Mdn={np.median(store_b)})', bins=6, edgecolor='black')
axes[1].legend()
axes[1].set_xlabel('Satisfaction Score')
axes[1].set_title('Histogram Overlay')

plt.tight_layout()
plt.show()

# 6. Wilcoxon Signed-Rank Test

---

## Overview
The Wilcoxon signed-rank test is the non-parametric equivalent of the **paired samples t-test**. It tests whether the median difference between paired observations is zero.

## Hypotheses

$$H_0: \text{The median of the differences } D_i = X_i - Y_i \text{ is zero}$$

$$H_1: \text{The median of } D_i \neq 0$$

## Assumptions
1. Data consists of **matched pairs** (before/after, left/right, etc.)
2. The differences $$D_i$$ are **continuous** and **symmetric** about the median
3. The differences are **independent** of each other
4. Measurement scale is at least **interval** for the differences

## Mathematical Formulation

### Step 1: Compute Differences
$$D_i = X_i - Y_i \quad \text{for } i = 1, 2, \ldots, n$$

### Step 2: Exclude Zero Differences
Remove pairs where $$D_i = 0$$. Let $$n_r$$ be the remaining count.

### Step 3: Rank Absolute Differences
Rank $$|D_1|, |D_2|, \ldots, |D_{n_r}|$$ from smallest to largest.

### Step 4: Compute Signed Rank Sums
$$W^+ = \sum_{D_i > 0} R_i \quad \text{(sum of ranks for positive differences)}$$

$$W^- = \sum_{D_i < 0} R_i \quad \text{(sum of ranks for negative differences)}$$

Note: $$W^+ + W^- = \frac{n_r(n_r + 1)}{2}$$

### Step 5: Test Statistic
$$W = \min(W^+, W^-)$$

### Step 6: Large-Sample Approximation ($$n_r > 25$$)

$$\mu_W = \frac{n_r(n_r + 1)}{4}$$

$$\sigma_W = \sqrt{\frac{n_r(n_r + 1)(2n_r + 1)}{24}}$$

$$z = \frac{W - \mu_W}{\sigma_W}$$

With **tied ranks correction**:

$$\sigma_W = \sqrt{\frac{n_r(n_r + 1)(2n_r + 1)}{24} - \sum_{k=1}^{g} \frac{t_k^3 - t_k}{48}}$$

## Effect Size: Matched-Pairs Rank-Biserial Correlation

$$r = \frac{W^+ - W^-}{W^+ + W^-} = \frac{2W^+}{n_r(n_r+1)} - 1$$

Alternatively: $$r = \frac{z}{\sqrt{n_r}}$$

## Interpretation of Results
- If $$p < \alpha$$: The median difference is significantly different from zero
- Direction: If $$W^+ > W^-$$, the post-treatment values tend to be larger
- Effect size $$r$$: 0.1 = small, 0.3 = medium, 0.5 = large (Cohen's benchmarks)

## Industrial Applications
- **Clinical Trials**: Before/after treatment comparisons on ordinal pain scales
- **UX Research**: Comparing user task completion times before/after interface redesign
- **Education**: Pre-test vs post-test scores for training effectiveness
- **Quality Control**: Measurements by two different instruments on the same items
- **Marketing**: Customer ratings before/after a campaign intervention

In [0]:
# =============================================================================
# WILCOXON SIGNED-RANK TEST - Complete Working Example
# =============================================================================

# Scenario: Employee productivity scores before and after a new tool is introduced
before = np.array([45, 52, 48, 61, 55, 43, 58, 47, 50, 53, 49, 56, 44, 51, 60])
after = np.array([52, 58, 50, 65, 60, 48, 62, 53, 55, 57, 54, 61, 49, 56, 63])

print("=" * 70)
print("WILCOXON SIGNED-RANK TEST")
print("=" * 70)
print(f"\nBefore: n={len(before)}, median={np.median(before)}, mean={before.mean():.2f}")
print(f"After:  n={len(after)}, median={np.median(after)}, mean={after.mean():.2f}")

# Compute differences
differences = after - before
print(f"\nDifferences (After - Before): {differences}")
print(f"Median difference: {np.median(differences)}")

# Perform the test
w_stat, p_value = stats.wilcoxon(before, after, alternative='two-sided')

print(f"\n--- Results ---")
print(f"W statistic: {w_stat}")
print(f"P-value: {p_value:.6f}")

# Manual computation
print(f"\n--- Manual Step-by-Step ---")
# Remove zero differences
non_zero_diff = differences[differences != 0]
n_r = len(non_zero_diff)
print(f"Non-zero differences: {non_zero_diff} (n_r = {n_r})")

# Rank absolute differences
abs_diff = np.abs(non_zero_diff)
ranks = stats.rankdata(abs_diff)
print(f"Absolute differences: {abs_diff}")
print(f"Ranks: {ranks}")

# Signed rank sums
W_plus = ranks[non_zero_diff > 0].sum()
W_minus = ranks[non_zero_diff < 0].sum()
print(f"\nW+ (positive ranks sum): {W_plus}")
print(f"W- (negative ranks sum): {W_minus}")
print(f"W+ + W- = {W_plus + W_minus} should equal n_r(n_r+1)/2 = {n_r*(n_r+1)/2}")
print(f"W = min(W+, W-) = {min(W_plus, W_minus)}")

# Effect size
r_effect = (W_plus - W_minus) / (W_plus + W_minus)
print(f"\nEffect size (r): {r_effect:.4f}")
print(f"  Interpretation: {'Small' if abs(r_effect) < 0.3 else 'Medium' if abs(r_effect) < 0.5 else 'Large'} effect")

# Large-sample Z approximation
mu_W = n_r * (n_r + 1) / 4
sigma_W = np.sqrt(n_r * (n_r + 1) * (2 * n_r + 1) / 24)
z = (min(W_plus, W_minus) - mu_W) / sigma_W
print(f"\nZ-approximation: z = {z:.4f}")

# Interpretation
alpha = 0.05
print(f"\n--- Interpretation (α = {alpha}) ---")
if p_value < alpha:
    print(f"✓ REJECT H₀: Significant median difference (p={p_value:.6f} < {alpha})")
    print(f"  The new tool significantly {'improved' if np.median(differences) > 0 else 'decreased'} productivity.")
else:
    print(f"✗ FAIL TO REJECT H₀: No significant difference (p={p_value:.6f} ≥ {alpha})")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Paired comparison
for i in range(len(before)):
    axes[0].plot([0, 1], [before[i], after[i]], 'b-o', alpha=0.5, markersize=4)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Before', 'After'])
axes[0].set_ylabel('Productivity Score')
axes[0].set_title('Paired Observations')

# Differences distribution
axes[1].hist(differences, bins=8, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(0, color='red', linestyle='--', label='Zero')
axes[1].axvline(np.median(differences), color='green', linestyle='-', label=f'Median={np.median(differences)}')
axes[1].legend()
axes[1].set_xlabel('Difference (After - Before)')
axes[1].set_title('Distribution of Differences')

# Signed ranks
colors = ['green' if d > 0 else 'red' for d in non_zero_diff]
axes[2].bar(range(n_r), ranks * np.sign(non_zero_diff), color=colors, edgecolor='black', alpha=0.7)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_xlabel('Observation')
axes[2].set_ylabel('Signed Rank')
axes[2].set_title('Signed Ranks (Green=Positive, Red=Negative)')

plt.tight_layout()
plt.show()

# 7. Kruskal-Wallis H Test

---

## Overview
The Kruskal-Wallis H test is the non-parametric equivalent of **one-way ANOVA**. It tests whether $$k \geq 3$$ independent groups come from the same distribution (i.e., have the same median).

## Hypotheses

$$H_0: \theta_1 = \theta_2 = \cdots = \theta_k \quad \text{(all group medians are equal)}$$

$$H_1: \text{At least one } \theta_i \text{ differs from the others}$$

## Assumptions
1. $$k \geq 2$$ **independent** groups
2. Observations within and between groups are **independent**
3. The dependent variable is at least **ordinal**
4. All groups have similarly shaped distributions (under $$H_0$$)

## Mathematical Formulation

### Step 1: Combine all $$N = \sum_{i=1}^{k} n_i$$ observations and rank them.

### Step 2: Compute the H statistic

$$H = \frac{12}{N(N+1)} \sum_{i=1}^{k} \frac{R_i^2}{n_i} - 3(N+1)$$

where $$R_i$$ = sum of ranks in group $$i$$, $$n_i$$ = sample size of group $$i$$.

### Step 3: Tied Ranks Correction

$$H_{corrected} = \frac{H}{1 - \frac{\sum_{j=1}^{g}(t_j^3 - t_j)}{N^3 - N}}$$

where $$g$$ = number of tied groups, $$t_j$$ = number of tied observations in group $$j$$.

### Step 4: Distribution

Under $$H_0$$, $$H \sim \chi^2_{k-1}$$ (approximately, for large samples).

## Post-Hoc Tests

If $$H$$ is significant, use **Dunn's test** for pairwise comparisons:

$$z_{ij} = \frac{\bar{R}_i - \bar{R}_j}{\sqrt{\frac{N(N+1)}{12} \left( \frac{1}{n_i} + \frac{1}{n_j} \right)}}$$

Apply Bonferroni or Holm correction for multiple comparisons:

$$\alpha_{adjusted} = \frac{\alpha}{\binom{k}{2}}$$

## Effect Size: Epsilon-Squared

$$\epsilon^2 = \frac{H - k + 1}{N - k}$$

Alternatively, **eta-squared** based on H:

$$\eta^2_H = \frac{H - k + 1}{N - 1}$$

## Industrial Applications
- **Marketing**: Comparing customer satisfaction across $$k$$ different advertising channels
- **Manufacturing**: Comparing defect rates across multiple production shifts
- **Healthcare**: Comparing recovery times across 3+ treatment protocols
- **Agriculture**: Comparing crop yields across different fertilizer types
- **Software Engineering**: Comparing bug resolution times across teams

In [0]:
# =============================================================================
# KRUSKAL-WALLIS H TEST - Complete Working Example
# =============================================================================

# Scenario: Comparing delivery times (hours) across three shipping methods
standard = np.array([72, 78, 65, 80, 85, 70, 75, 90, 68, 82])
express = np.array([28, 35, 30, 25, 32, 40, 27, 33, 29, 31])
same_day = np.array([4, 6, 5, 8, 3, 7, 5, 6, 4, 7])

print("=" * 70)
print("KRUSKAL-WALLIS H TEST")
print("=" * 70)
print(f"\nStandard:  n={len(standard)}, median={np.median(standard):.1f}h")
print(f"Express:   n={len(express)}, median={np.median(express):.1f}h")
print(f"Same-day:  n={len(same_day)}, median={np.median(same_day):.1f}h")

# Perform the test
h_stat, p_value = stats.kruskal(standard, express, same_day)

print(f"\n--- Results ---")
print(f"H statistic: {h_stat:.4f}")
print(f"Degrees of freedom: {3 - 1}")
print(f"P-value: {p_value:.8f}")

# Effect size
N = len(standard) + len(express) + len(same_day)
k = 3
epsilon_sq = (h_stat - k + 1) / (N - k)
eta_sq = (h_stat - k + 1) / (N - 1)
print(f"\nEffect sizes:")
print(f"  ε² (epsilon-squared): {epsilon_sq:.4f}")
print(f"  η²_H (eta-squared): {eta_sq:.4f}")

# Manual computation
print(f"\n--- Manual Computation ---")
combined = np.concatenate([standard, express, same_day])
ranks = stats.rankdata(combined)
n1, n2, n3 = len(standard), len(express), len(same_day)

R1 = ranks[:n1].sum()
R2 = ranks[n1:n1+n2].sum()
R3 = ranks[n1+n2:].sum()

print(f"R1 (Standard) = {R1:.1f}")
print(f"R2 (Express)  = {R2:.1f}")
print(f"R3 (Same-day) = {R3:.1f}")

H_manual = (12 / (N * (N + 1))) * (R1**2/n1 + R2**2/n2 + R3**2/n3) - 3*(N+1)
print(f"H (manual) = {H_manual:.4f}")

# Interpretation
alpha = 0.05
print(f"\n--- Interpretation (α = {alpha}) ---")
if p_value < alpha:
    print(f"✓ REJECT H₀: Significant difference among groups (p={p_value:.2e} < {alpha})")
    print(f"  At least one shipping method has a significantly different delivery time.")
    
    # Post-hoc: Dunn's test (using scikit-posthocs)
    print(f"\n--- Post-Hoc: Pairwise Mann-Whitney with Bonferroni Correction ---")
    pairs = [('Standard vs Express', standard, express),
             ('Standard vs Same-day', standard, same_day),
             ('Express vs Same-day', express, same_day)]
    
    alpha_bonf = alpha / 3  # Bonferroni correction for 3 comparisons
    print(f"Bonferroni-adjusted α = {alpha_bonf:.4f}")
    
    for name, g1, g2 in pairs:
        u, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
        sig = "*" if p < alpha_bonf else "ns"
        print(f"  {name}: U={u:.1f}, p={p:.6f} {sig}")
else:
    print(f"✗ FAIL TO REJECT H₀: No significant difference (p={p_value:.4f} ≥ {alpha})")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot([standard, express, same_day], labels=['Standard', 'Express', 'Same-day'])
axes[0].set_ylabel('Delivery Time (hours)')
axes[0].set_title(f'Kruskal-Wallis: H={h_stat:.2f}, p={p_value:.2e}')

# Rank distributions
axes[1].bar(['Standard', 'Express', 'Same-day'], 
            [R1/n1, R2/n2, R3/n3], 
            color=['#2196F3', '#FF9800', '#4CAF50'], edgecolor='black', alpha=0.7)
axes[1].axhline(y=(N+1)/2, color='red', linestyle='--', label=f'Expected mean rank = {(N+1)/2:.1f}')
axes[1].set_ylabel('Mean Rank')
axes[1].set_title('Mean Ranks by Group')
axes[1].legend()

plt.tight_layout()
plt.show()

# 8. Friedman Test

---

## Overview
The Friedman test is the non-parametric equivalent of **repeated-measures ANOVA** (or two-way ANOVA without replication). It tests whether $$k \geq 3$$ related groups (repeated measurements on the same subjects) have the same distribution.

## Hypotheses

$$H_0: \text{The distributions of all } k \text{ treatments are identical}$$

$$H_1: \text{At least one treatment distribution differs}$$

## Assumptions
1. One group of $$n$$ subjects measured under $$k$$ conditions
2. The dependent variable is at least **ordinal**
3. Observations across subjects are **independent** (within-subject dependence is expected)
4. No interaction between subjects and treatments (additivity)

## Mathematical Formulation

### Step 1: Rank Within Each Subject
For each subject $$i$$, rank the $$k$$ measurements from 1 to $$k$$.

### Step 2: Compute Rank Sums
$$R_j = \sum_{i=1}^{n} r_{ij} \quad \text{for each treatment } j = 1, \ldots, k$$

where $$r_{ij}$$ is the rank of treatment $$j$$ for subject $$i$$.

### Step 3: Friedman Statistic

$$\chi^2_F = \frac{12}{nk(k+1)} \sum_{j=1}^{k} R_j^2 - 3n(k+1)$$

Alternative (equivalent) form:

$$\chi^2_F = \frac{12}{nk(k+1)} \left[ \sum_{j=1}^{k} R_j^2 \right] - 3n(k+1)$$

### Step 4: Distribution

Under $$H_0$$: $$\chi^2_F \sim \chi^2_{k-1}$$ approximately (exact tables exist for small $$n, k$$).

### Improved Statistic (Iman-Davenport)

For better accuracy:

$$F_{ID} = \frac{(n-1) \chi^2_F}{n(k-1) - \chi^2_F}$$

which follows $$F_{(k-1), (k-1)(n-1)}$$ distribution.

## Post-Hoc: Nemenyi Test

Critical difference:

$$CD = q_{\alpha} \sqrt{\frac{k(k+1)}{6n}}$$

where $$q_{\alpha}$$ is from the Studentized range distribution.

Two treatments differ significantly if:

$$|\bar{R}_i - \bar{R}_j| > CD$$

## Effect Size: Kendall's W (Coefficient of Concordance)

$$W = \frac{\chi^2_F}{n(k-1)} = \frac{12 \sum_{j=1}^k (R_j - \bar{R})^2}{n^2 k(k^2 - 1)}$$

Interpretation: $$W = 0$$ (no agreement), $$W = 1$$ (perfect agreement in ranking)

## Industrial Applications
- **Consumer Research**: Rating products by the same panel of judges across sessions
- **Software Testing**: Comparing $$k$$ algorithms on the same benchmark datasets
- **Medical Research**: Comparing $$k$$ treatments on the same patients over time
- **Education**: Comparing teaching methods where same students experience all methods
- **Sports Analytics**: Ranking performance of athletes across multiple competitions

In [0]:
# =============================================================================
# FRIEDMAN TEST - Complete Working Example
# =============================================================================

# Scenario: 12 judges rate 4 wine varieties (scores 1-100)
np.random.seed(123)
n_judges = 12

# Simulated wine ratings (same judges rate all wines)
wine_a = np.array([78, 82, 75, 80, 85, 77, 79, 83, 76, 81, 84, 78])
wine_b = np.array([85, 88, 82, 87, 90, 84, 86, 89, 83, 88, 91, 85])
wine_c = np.array([72, 76, 70, 74, 78, 71, 73, 77, 69, 75, 79, 73])
wine_d = np.array([80, 84, 78, 82, 86, 79, 81, 85, 77, 83, 87, 80])

print("=" * 70)
print("FRIEDMAN TEST")
print("=" * 70)
print(f"\n{'Wine':<10} {'Median':<10} {'Mean':<10} {'Std':<10}")
for name, data in [('A', wine_a), ('B', wine_b), ('C', wine_c), ('D', wine_d)]:
    print(f"{name:<10} {np.median(data):<10.1f} {data.mean():<10.2f} {data.std():<10.2f}")

# Perform the test
f_stat, p_value = stats.friedmanchisquare(wine_a, wine_b, wine_c, wine_d)

print(f"\n--- Results ---")
print(f"Friedman χ² statistic: {f_stat:.4f}")
print(f"Degrees of freedom: {4 - 1}")
print(f"P-value: {p_value:.8f}")

# Effect size: Kendall's W
k = 4
W = f_stat / (n_judges * (k - 1))
print(f"\nKendall's W (concordance): {W:.4f}")
print(f"  Interpretation: {'Weak' if W < 0.3 else 'Moderate' if W < 0.7 else 'Strong'} agreement among judges")

# Manual computation
print(f"\n--- Manual Computation ---")
data_matrix = np.column_stack([wine_a, wine_b, wine_c, wine_d])

# Rank within each row (judge)
ranks_matrix = np.zeros_like(data_matrix, dtype=float)
for i in range(n_judges):
    ranks_matrix[i] = stats.rankdata(data_matrix[i])

print("\nRank matrix (first 5 judges):")
print(pd.DataFrame(ranks_matrix[:5], columns=['Wine A', 'Wine B', 'Wine C', 'Wine D'],
                   index=[f'Judge {i+1}' for i in range(5)]))

# Column rank sums
R = ranks_matrix.sum(axis=0)
print(f"\nRank sums: R_A={R[0]:.0f}, R_B={R[1]:.0f}, R_C={R[2]:.0f}, R_D={R[3]:.0f}")
print(f"Mean ranks: {R/n_judges}")

# Compute chi-square
chi2_manual = (12 / (n_judges * k * (k + 1))) * np.sum(R**2) - 3 * n_judges * (k + 1)
print(f"χ² (manual) = {chi2_manual:.4f}")

# Post-hoc pairwise comparisons (Wilcoxon signed-rank with Bonferroni)
alpha = 0.05
print(f"\n--- Post-Hoc: Pairwise Wilcoxon Signed-Rank (Bonferroni) ---")
wines = {'A': wine_a, 'B': wine_b, 'C': wine_c, 'D': wine_d}
from itertools import combinations

num_comparisons = len(list(combinations(wines.keys(), 2)))
alpha_bonf = alpha / num_comparisons
print(f"Number of comparisons: {num_comparisons}, Bonferroni α = {alpha_bonf:.4f}")

for (n1, g1), (n2, g2) in combinations(wines.items(), 2):
    w, p = stats.wilcoxon(g1, g2)
    sig = "*" if p < alpha_bonf else "ns"
    print(f"  {n1} vs {n2}: W={w:.0f}, p={p:.6f} {sig}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Mean ranks plot
mean_ranks = R / n_judges
colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']
axes[0].bar(['Wine A', 'Wine B', 'Wine C', 'Wine D'], mean_ranks, color=colors, edgecolor='black')
axes[0].axhline(y=(k+1)/2, color='gray', linestyle='--', label=f'Expected under H₀ = {(k+1)/2:.1f}')
axes[0].set_ylabel('Mean Rank')
axes[0].set_title(f'Friedman Test: χ²={f_stat:.2f}, p={p_value:.4e}')
axes[0].legend()

# Individual judge profiles
for i in range(n_judges):
    axes[1].plot(['A', 'B', 'C', 'D'], data_matrix[i], 'o-', alpha=0.3, markersize=3)
axes[1].plot(['A', 'B', 'C', 'D'], data_matrix.mean(axis=0), 'ko-', linewidth=3, markersize=8, label='Mean')
axes[1].set_xlabel('Wine Variety')
axes[1].set_ylabel('Rating')
axes[1].set_title('Individual Judge Profiles')
axes[1].legend()

plt.tight_layout()
plt.show()

# 9. Chi-Square Tests

---

## 9.1 Chi-Square Goodness-of-Fit Test

### Overview
Tests whether observed frequencies in categories differ significantly from expected (theoretical) frequencies.

### Hypotheses

$$H_0: O_i = E_i \text{ for all categories } i$$

$$H_1: O_i \neq E_i \text{ for at least one category}$$

### Test Statistic

$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}$$

where $$O_i$$ = observed frequency, $$E_i$$ = expected frequency, $$k$$ = number of categories.

Under $$H_0$$: $$\chi^2 \sim \chi^2_{k-1-m}$$ where $$m$$ = number of estimated parameters.

### Assumptions
1. Observations are **independent**
2. Categories are **mutually exclusive** and **exhaustive**
3. $$E_i \geq 5$$ for all categories (rule of thumb; merge categories if violated)
4. Data is **frequency/count** data

---

## 9.2 Chi-Square Test of Independence

### Overview
Tests whether two categorical variables are associated or independent in a contingency table.

### Hypotheses

$$H_0: P(A_i \cap B_j) = P(A_i) \cdot P(B_j) \text{ for all } i, j \quad \text{(independence)}$$

$$H_1: \text{The variables are associated (not independent)}$$

### Test Statistic

For an $$r \times c$$ contingency table:

$$\chi^2 = \sum_{i=1}^{r} \sum_{j=1}^{c} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

where:

$$E_{ij} = \frac{R_i \cdot C_j}{N}$$

$$R_i$$ = row total, $$C_j$$ = column total, $$N$$ = grand total.

Degrees of freedom: $$df = (r-1)(c-1)$$

### Effect Sizes

**Cramér's V** (for any $$r \times c$$ table):

$$V = \sqrt{\frac{\chi^2}{N \cdot \min(r-1, c-1)}}$$

Interpretation: $$V \in [0, 1]$$; 0.1 = small, 0.3 = medium, 0.5 = large

**Phi coefficient** (for $$2 \times 2$$ tables only):

$$\phi = \sqrt{\frac{\chi^2}{N}}$$

**Odds Ratio** (for $$2 \times 2$$ tables):

$$OR = \frac{O_{11} \cdot O_{22}}{O_{12} \cdot O_{21}}$$

### Yates' Continuity Correction (for $$2 \times 2$$)

$$\chi^2_{Yates} = \sum \frac{(|O_{ij} - E_{ij}| - 0.5)^2}{E_{ij}}$$

Used when any $$E_{ij} < 10$$ in a $$2 \times 2$$ table.

### Fisher's Exact Test
When $$E_{ij} < 5$$ (small expected frequencies), use Fisher's exact test instead:

$$p = \frac{\binom{R_1}{O_{11}} \binom{R_2}{O_{21}}}{\binom{N}{C_1}} = \frac{R_1! \cdot R_2! \cdot C_1! \cdot C_2!}{N! \cdot O_{11}! \cdot O_{12}! \cdot O_{21}! \cdot O_{22}!}$$

## Industrial Applications
- **Market Research**: Testing whether product preference is independent of age group
- **Quality Control**: Testing whether defect type distribution matches historical patterns
- **Healthcare**: Testing association between treatment type and recovery outcome
- **A/B Testing**: Testing whether conversion rates differ across variants (categorical outcome)
- **Genetics**: Testing Hardy-Weinberg equilibrium in population genetics
- **Survey Analysis**: Testing independence of demographic variables and opinions

In [0]:
# =============================================================================
# CHI-SQUARE TESTS - Complete Working Examples
# =============================================================================

print("=" * 70)
print("CHI-SQUARE GOODNESS-OF-FIT TEST")
print("=" * 70)

# Scenario: A die is rolled 120 times. Is it fair?
observed = np.array([25, 17, 15, 23, 24, 16])  # Observed frequencies
expected = np.array([20, 20, 20, 20, 20, 20])   # Expected if fair

print(f"Observed: {observed} (sum={observed.sum()})")
print(f"Expected: {expected} (sum={expected.sum()})")

chi2_stat, p_value = stats.chisquare(observed, f_exp=expected)

print(f"\nχ² statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {len(observed) - 1}")
print(f"P-value: {p_value:.4f}")

# Manual calculation
chi2_manual = np.sum((observed - expected)**2 / expected)
print(f"χ² (manual): {chi2_manual:.4f}")

alpha = 0.05
print(f"\nConclusion (α={alpha}): {'REJECT H₀ - Die is NOT fair' if p_value < alpha else 'FAIL TO REJECT H₀ - No evidence die is unfair'}")

print("\n" + "=" * 70)
print("CHI-SQUARE TEST OF INDEPENDENCE")
print("=" * 70)

# Scenario: Is there an association between customer segment and churn?
# Contingency table: rows=segment, cols=churned (Yes/No)
contingency = np.array([
    [45, 155],   # Premium: churned, not churned
    [80, 120],   # Standard: churned, not churned  
    [95, 105],   # Basic: churned, not churned
])

df_contingency = pd.DataFrame(
    contingency,
    index=['Premium', 'Standard', 'Basic'],
    columns=['Churned', 'Retained']
)
print("\nContingency Table:")
print(df_contingency)
print(f"\nTotal: {contingency.sum()}")

# Perform test
chi2, p_val, dof, expected_freq = stats.chi2_contingency(contingency)

print(f"\n--- Results ---")
print(f"χ² statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_val:.6f}")

# Expected frequencies
print(f"\nExpected frequencies:")
print(pd.DataFrame(expected_freq.round(2), 
                   index=['Premium', 'Standard', 'Basic'],
                   columns=['Churned', 'Retained']))

# Check assumption: all expected >= 5
print(f"\nAll expected ≥ 5? {(expected_freq >= 5).all()} (min = {expected_freq.min():.1f})")

# Effect size: Cramér's V
N = contingency.sum()
r, c = contingency.shape
cramers_v = np.sqrt(chi2 / (N * min(r-1, c-1)))
print(f"\nCramér's V: {cramers_v:.4f}")
print(f"  Interpretation: {'Small' if cramers_v < 0.3 else 'Medium' if cramers_v < 0.5 else 'Large'} effect")

# Standardized residuals (shows which cells drive the significance)
std_residuals = (contingency - expected_freq) / np.sqrt(expected_freq)
print(f"\nStandardized Residuals (|value| > 2 is significant):")
print(pd.DataFrame(std_residuals.round(3),
                   index=['Premium', 'Standard', 'Basic'],
                   columns=['Churned', 'Retained']))

print(f"\n--- Interpretation (α={alpha}) ---")
if p_val < alpha:
    print(f"✓ REJECT H₀: Significant association between segment and churn (p={p_val:.4f})")
    print(f"  Premium customers churn LESS than expected (residual={std_residuals[0,0]:.2f})")
    print(f"  Basic customers churn MORE than expected (residual={std_residuals[2,0]:.2f})")
else:
    print(f"✗ FAIL TO REJECT H₀: No significant association (p={p_val:.4f})")

# --- Fisher's Exact Test (2x2 example) ---
print("\n" + "=" * 70)
print("FISHER'S EXACT TEST (2×2 table)")
print("=" * 70)

# Small sample: Drug vs Placebo outcome
table_2x2 = np.array([[8, 2],    # Drug: cured, not cured
                      [3, 7]])   # Placebo: cured, not cured
print("\n           Cured  Not Cured")
print(f"Drug       {table_2x2[0,0]}      {table_2x2[0,1]}")
print(f"Placebo    {table_2x2[1,0]}      {table_2x2[1,1]}")

odds_ratio, p_fisher = stats.fisher_exact(table_2x2)
print(f"\nOdds Ratio: {odds_ratio:.4f}")
print(f"P-value (Fisher's exact): {p_fisher:.4f}")
print(f"Interpretation: Drug patients are {odds_ratio:.1f}x more likely to be cured.")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Stacked bar chart for independence test
df_plot = df_contingency.div(df_contingency.sum(axis=1), axis=0) * 100
df_plot.plot(kind='bar', stacked=True, ax=axes[0], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_title(f'Churn by Segment (χ²={chi2:.2f}, p={p_val:.4f})')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(title='Status')

# Residuals heatmap
im = axes[1].imshow(std_residuals, cmap='RdBu_r', vmin=-3, vmax=3, aspect='auto')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Churned', 'Retained'])
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(['Premium', 'Standard', 'Basic'])
for i in range(3):
    for j in range(2):
        axes[1].text(j, i, f'{std_residuals[i,j]:.2f}', ha='center', va='center', fontsize=12)
axes[1].set_title('Standardized Residuals')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

# 10. Kolmogorov-Smirnov (KS) Test

---

## Overview
The KS test compares the empirical cumulative distribution function (ECDF) of a sample against a reference distribution (one-sample) or against another sample's ECDF (two-sample). It is sensitive to **any** difference in distributions — location, spread, and shape.

## 10.1 One-Sample KS Test

### Hypotheses

$$H_0: F(x) = F_0(x) \text{ for all } x \quad \text{(sample comes from reference distribution)}$$

$$H_1: F(x) \neq F_0(x) \text{ for some } x$$

### Test Statistic

$$D_n = \sup_x |F_n(x) - F_0(x)|$$

where $$F_n(x) = \frac{1}{n} \sum_{i=1}^{n} \mathbf{1}(X_i \leq x)$$ is the ECDF.

In practice:

$$D_n = \max_{1 \leq i \leq n} \left\{ \max\left( \left|\frac{i}{n} - F_0(X_{(i)})\right|, \left|\frac{i-1}{n} - F_0(X_{(i)})\right| \right) \right\}$$

where $$X_{(1)} \leq X_{(2)} \leq \cdots \leq X_{(n)}$$ are order statistics.

## 10.2 Two-Sample KS Test

### Test Statistic

$$D_{n,m} = \sup_x |F_n(x) - G_m(x)|$$

where $$F_n$$ and $$G_m$$ are the ECDFs of the two samples.

### Critical Value (Asymptotic)

Reject $$H_0$$ if:

$$D_{n,m} > c(\alpha) \sqrt{\frac{n + m}{nm}}$$

where $$c(\alpha) = \sqrt{-\frac{1}{2} \ln\left(\frac{\alpha}{2}\right)}$$

For $$\alpha = 0.05$$: $$c(0.05) \approx 1.358$$

## Assumptions
1. Data is **continuous** (the test is conservative for discrete data)
2. Observations are **independent**
3. For one-sample: the reference distribution is **fully specified** (parameters NOT estimated from data)
4. If parameters are estimated from data, use the **Lilliefors test** instead

## Strengths and Limitations

| Strengths | Limitations |
|-----------|-------------|
| Distribution-free under $$H_0$$ | Less powerful than Anderson-Darling for tails |
| Detects any distributional difference | Designed for continuous distributions |
| Exact p-values available | Not valid if distribution parameters estimated from data |
| No binning required | Less sensitive to tail differences |

## Industrial Applications
- **Finance**: Testing whether stock returns follow a normal distribution
- **Reliability Engineering**: Testing whether failure times follow a Weibull distribution
- **Data Quality**: Detecting distribution drift in production ML models
- **Simulation**: Validating random number generators
- **Environmental Science**: Comparing pollution distributions across regions

In [0]:
# =============================================================================
# KOLMOGOROV-SMIRNOV TEST - Complete Working Examples
# =============================================================================

np.random.seed(42)

print("=" * 70)
print("ONE-SAMPLE KS TEST: Testing for Normality")
print("=" * 70)

# Scenario: Are these response times normally distributed?
response_times = np.concatenate([
    np.random.normal(200, 30, 80),
    np.random.exponential(50, 20)  # Some skewed outliers
])

print(f"Sample: n={len(response_times)}, mean={response_times.mean():.2f}, std={response_times.std():.2f}")
print(f"Skewness: {stats.skew(response_times):.4f}, Kurtosis: {stats.kurtosis(response_times):.4f}")

# Test against normal distribution with sample mean and std
# Note: This is technically a Lilliefors situation, but we demonstrate the API
ks_stat, p_value = stats.kstest(response_times, 'norm', 
                                args=(response_times.mean(), response_times.std()))

print(f"\n--- Results ---")
print(f"KS statistic (D): {ks_stat:.4f}")
print(f"P-value: {p_value:.4f}")

alpha = 0.05
print(f"\nConclusion (α={alpha}): {'REJECT H₀ - Data is NOT normal' if p_value < alpha else 'FAIL TO REJECT H₀ - Consistent with normality'}")

# Also run Shapiro-Wilk for comparison (more powerful normality test)
shapiro_stat, shapiro_p = stats.shapiro(response_times)
print(f"\nShapiro-Wilk comparison: W={shapiro_stat:.4f}, p={shapiro_p:.4f}")

print("\n" + "=" * 70)
print("TWO-SAMPLE KS TEST: Comparing Distributions")
print("=" * 70)

# Scenario: Has the distribution of page load times changed after deployment?
before_deploy = np.random.gamma(4, 0.5, 150)   # Before: Gamma(4, 0.5)
after_deploy = np.random.gamma(3, 0.4, 120)    # After: Slightly different

print(f"\nBefore deployment: n={len(before_deploy)}, median={np.median(before_deploy):.3f}")
print(f"After deployment:  n={len(after_deploy)}, median={np.median(after_deploy):.3f}")

ks_stat_2, p_value_2 = stats.ks_2samp(before_deploy, after_deploy)

print(f"\n--- Results ---")
print(f"KS statistic (D): {ks_stat_2:.4f}")
print(f"P-value: {p_value_2:.6f}")
print(f"\nConclusion: {'REJECT H₀ - Distributions differ' if p_value_2 < alpha else 'FAIL TO REJECT H₀ - No evidence of distribution change'}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# One-sample: Histogram + theoretical normal
x_range = np.linspace(response_times.min(), response_times.max(), 100)
axes[0, 0].hist(response_times, bins=20, density=True, alpha=0.7, edgecolor='black', label='Data')
axes[0, 0].plot(x_range, stats.norm.pdf(x_range, response_times.mean(), response_times.std()),
               'r-', linewidth=2, label='Normal fit')
axes[0, 0].set_title(f'One-Sample: Data vs Normal (D={ks_stat:.3f})')
axes[0, 0].legend()
axes[0, 0].set_xlabel('Response Time (ms)')

# One-sample: ECDF comparison
sorted_data = np.sort(response_times)
ecdf = np.arange(1, len(sorted_data)+1) / len(sorted_data)
theoretical_cdf = stats.norm.cdf(sorted_data, response_times.mean(), response_times.std())

axes[0, 1].step(sorted_data, ecdf, label='ECDF', linewidth=2)
axes[0, 1].plot(sorted_data, theoretical_cdf, 'r-', label='Normal CDF', linewidth=2)
# Show max deviation
max_idx = np.argmax(np.abs(ecdf - theoretical_cdf))
axes[0, 1].vlines(sorted_data[max_idx], theoretical_cdf[max_idx], ecdf[max_idx],
                  color='green', linewidth=3, label=f'D = {ks_stat:.3f}')
axes[0, 1].set_title('ECDF vs Theoretical CDF')
axes[0, 1].legend()
axes[0, 1].set_xlabel('Response Time (ms)')

# Two-sample: Overlaid histograms
axes[1, 0].hist(before_deploy, bins=20, density=True, alpha=0.6, label='Before', edgecolor='black')
axes[1, 0].hist(after_deploy, bins=20, density=True, alpha=0.6, label='After', edgecolor='black')
axes[1, 0].set_title(f'Two-Sample: Before vs After (D={ks_stat_2:.3f})')
axes[1, 0].legend()
axes[1, 0].set_xlabel('Page Load Time (s)')

# Two-sample: ECDF comparison
sorted_before = np.sort(before_deploy)
sorted_after = np.sort(after_deploy)
ecdf_before = np.arange(1, len(sorted_before)+1) / len(sorted_before)
ecdf_after = np.arange(1, len(sorted_after)+1) / len(sorted_after)

axes[1, 1].step(sorted_before, ecdf_before, label='Before', linewidth=2)
axes[1, 1].step(sorted_after, ecdf_after, label='After', linewidth=2)
axes[1, 1].set_title('ECDFs Comparison')
axes[1, 1].legend()
axes[1, 1].set_xlabel('Page Load Time (s)')
axes[1, 1].set_ylabel('Cumulative Probability')

plt.tight_layout()
plt.show()

# 11. Spearman's Rank Correlation

---

## Overview
Spearman's rank correlation coefficient ($$\rho_s$$ or $$r_s$$) measures the **monotonic** relationship between two variables. It is the non-parametric equivalent of **Pearson's correlation** and works on ranks rather than raw values.

## Mathematical Formulation

### Method 1: Pearson Correlation on Ranks

$$\rho_s = \frac{\sum_{i=1}^n (R(x_i) - \bar{R}_x)(R(y_i) - \bar{R}_y)}{\sqrt{\sum_{i=1}^n (R(x_i) - \bar{R}_x)^2 \cdot \sum_{i=1}^n (R(y_i) - \bar{R}_y)^2}}$$

where $$R(x_i)$$ = rank of $$x_i$$, $$\bar{R}_x = \frac{n+1}{2}$$.

### Method 2: Shortcut Formula (No Ties)

When there are **no tied ranks**:

$$\rho_s = 1 - \frac{6 \sum_{i=1}^n d_i^2}{n(n^2 - 1)}$$

where $$d_i = R(x_i) - R(y_i)$$ (difference in ranks for observation $$i$$).

### Hypothesis Testing

$$H_0: \rho_s = 0 \quad \text{(no monotonic association)}$$

For $$n > 10$$, the test statistic:

$$t = \rho_s \sqrt{\frac{n - 2}{1 - \rho_s^2}}$$

follows a $$t_{n-2}$$ distribution under $$H_0$$.

## Interpretation

| $$\rho_s$$ | Interpretation |
|---|---|
| $$1.0$$ | Perfect positive monotonic relationship |
| $$0.7$$ to $$1.0$$ | Strong positive |
| $$0.4$$ to $$0.7$$ | Moderate positive |
| $$0.1$$ to $$0.4$$ | Weak positive |
| $$0.0$$ | No monotonic relationship |
| $$-1.0$$ | Perfect negative monotonic relationship |

## Key Difference from Pearson's $$r$$
- **Pearson** measures **linear** relationships: $$y = ax + b$$
- **Spearman** measures **monotonic** relationships: if $$x$$ increases, $$y$$ always increases (or always decreases)
- A relationship can be perfectly monotonic but non-linear (e.g., exponential), giving $$\rho_s = 1$$ but $$r < 1$$

## Assumptions
1. Both variables are at least **ordinal**
2. Observations are **paired**
3. The relationship (if any) is **monotonic** (not necessarily linear)

## Kendall's Tau ($$\tau$$) — Related Measure

Another rank correlation:

$$\tau = \frac{C - D}{\binom{n}{2}} = \frac{2(C - D)}{n(n-1)}$$

where $$C$$ = concordant pairs, $$D$$ = discordant pairs.

A pair $$(x_i, y_i)$$ and $$(x_j, y_j)$$ is:
- **Concordant** if $$(x_i - x_j)(y_i - y_j) > 0$$
- **Discordant** if $$(x_i - x_j)(y_i - y_j) < 0$$

## Industrial Applications
- **Finance**: Correlation between credit scores and default rates (ordinal)
- **Education**: Correlation between class rank and test performance
- **Psychology**: Correlation between anxiety rating scales and performance
- **E-commerce**: Correlation between search ranking and click-through rate
- **Environmental Science**: Monotonic trends in pollution over time (Mann-Kendall trend test uses $$\tau$$)

In [0]:
# =============================================================================
# SPEARMAN'S RANK CORRELATION - Complete Working Example
# =============================================================================

np.random.seed(42)

print("=" * 70)
print("SPEARMAN'S RANK CORRELATION")
print("=" * 70)

# Scenario: Relationship between years of experience and salary
# (likely monotonic but non-linear due to diminishing returns)
n = 20
experience = np.array([1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 14, 15, 16, 18, 20, 22, 24, 25, 28, 30])
# Salary follows a log-like curve (monotonic but non-linear)
salary = 40000 + 15000 * np.log1p(experience) + np.random.normal(0, 3000, n)

print(f"Sample size: n = {n}")
print(f"Experience range: {experience.min()} - {experience.max()} years")
print(f"Salary range: ${salary.min():.0f} - ${salary.max():.0f}")

# Spearman correlation
rho_s, p_value_s = stats.spearmanr(experience, salary)

# Pearson correlation (for comparison)
r_p, p_value_p = stats.pearsonr(experience, salary)

# Kendall's tau (for comparison)
tau, p_value_tau = stats.kendalltau(experience, salary)

print(f"\n--- Results ---")
print(f"{'Measure':<25} {'Value':<12} {'P-value':<12}")
print(f"{'Spearman ρ_s':<25} {rho_s:<12.4f} {p_value_s:<12.6f}")
print(f"{'Pearson r':<25} {r_p:<12.4f} {p_value_p:<12.6f}")
print(f"{'Kendall τ':<25} {tau:<12.4f} {p_value_tau:<12.6f}")

# Manual computation (shortcut formula)
print(f"\n--- Manual Computation (Shortcut Formula) ---")
ranks_x = stats.rankdata(experience)
ranks_y = stats.rankdata(salary)
d = ranks_x - ranks_y
d_squared = d**2

print(f"Rank differences (d): {d[:10].astype(int)}...")
print(f"Sum of d²: {d_squared.sum():.2f}")

rho_manual = 1 - (6 * d_squared.sum()) / (n * (n**2 - 1))
print(f"ρ_s (manual, shortcut): {rho_manual:.4f}")
print(f"ρ_s (scipy):           {rho_s:.4f}")
print(f"Note: Small difference due to tied ranks (shortcut assumes no ties)")

# T-test for significance
t_stat = rho_s * np.sqrt((n - 2) / (1 - rho_s**2))
print(f"\nt-statistic: {t_stat:.4f}")
print(f"P-value (manual): {2 * (1 - stats.t.cdf(abs(t_stat), df=n-2)):.6f}")

# Interpretation
alpha = 0.05
print(f"\n--- Interpretation (α={alpha}) ---")
if p_value_s < alpha:
    strength = 'Strong' if abs(rho_s) >= 0.7 else 'Moderate' if abs(rho_s) >= 0.4 else 'Weak'
    direction = 'positive' if rho_s > 0 else 'negative'
    print(f"✓ Significant {strength.lower()} {direction} monotonic relationship (ρ_s={rho_s:.3f}, p={p_value_s:.4f})")
    print(f"  As experience increases, salary monotonically increases.")
    print(f"  Note: Spearman ({rho_s:.3f}) > Pearson ({r_p:.3f}) → relationship is monotonic but non-linear.")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Scatter plot with both fits
axes[0].scatter(experience, salary/1000, alpha=0.7, edgecolor='black', s=60)
# Linear fit (Pearson)
z = np.polyfit(experience, salary/1000, 1)
axes[0].plot(experience, np.polyval(z, experience), 'r--', label=f'Linear (r={r_p:.3f})', linewidth=2)
# Log fit (better for monotonic)
log_fit = np.polyfit(np.log1p(experience), salary/1000, 1)
axes[0].plot(np.sort(experience), np.polyval(log_fit, np.log1p(np.sort(experience))), 
            'g-', label=f'Log fit (ρ_s={rho_s:.3f})', linewidth=2)
axes[0].set_xlabel('Years of Experience')
axes[0].set_ylabel('Salary ($K)')
axes[0].set_title('Experience vs Salary')
axes[0].legend()

# Rank-rank plot
axes[1].scatter(ranks_x, ranks_y, alpha=0.7, edgecolor='black', s=60, color='purple')
axes[1].plot([0, n+1], [0, n+1], 'r--', alpha=0.5, label='Perfect agreement')
axes[1].set_xlabel('Rank (Experience)')
axes[1].set_ylabel('Rank (Salary)')
axes[1].set_title(f'Rank-Rank Plot (ρ_s = {rho_s:.3f})')
axes[1].legend()

# Comparison of correlations for different relationships
# Generate examples: linear, monotonic non-linear, non-monotonic
x_demo = np.linspace(0, 5, 50)
relationships = {
    'Linear: y=2x': (x_demo, 2*x_demo + np.random.normal(0, 0.5, 50)),
    'Monotonic: y=x³': (x_demo, x_demo**3 + np.random.normal(0, 5, 50)),
    'Non-monotonic: y=sin(x)': (x_demo, np.sin(x_demo) + np.random.normal(0, 0.1, 50))
}

for name, (x, y) in relationships.items():
    r, _ = stats.pearsonr(x, y)
    rho, _ = stats.spearmanr(x, y)
    axes[2].scatter([], [], label=f'{name.split(":")[0]}: r={r:.2f}, ρ={rho:.2f}')

axes[2].legend(loc='upper left', fontsize=9)
axes[2].set_title('Pearson vs Spearman Comparison')
axes[2].text(0.5, 0.4, 'Spearman captures\nmonotonic non-linear\nrelationships better\nthan Pearson', 
            transform=axes[2].transAxes, ha='center', fontsize=11, style='italic')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# 12. Sign Test

---

## Overview
The simplest non-parametric test for paired data. It only considers the **direction** (sign) of differences, ignoring magnitudes. It is the non-parametric alternative to the paired t-test when even the Wilcoxon signed-rank assumptions (symmetry) cannot be met.

## Hypotheses

$$H_0: P(X > Y) = P(X < Y) = 0.5 \quad \text{(median difference is zero)}$$

$$H_1: P(X > Y) \neq P(X < Y)$$

Equivalently: the probability of a positive difference equals the probability of a negative difference.

## Mathematical Formulation

### Step 1: Compute Differences
$$D_i = X_i - Y_i$$

### Step 2: Count Signs
Let:
- $$n^+$$ = number of positive differences
- $$n^-$$ = number of negative differences
- $$n_0$$ = number of zero differences (discarded)
- $$n = n^+ + n^-$$ (effective sample size)

### Step 3: Test Statistic
Under $$H_0$$, $$n^+ \sim \text{Binomial}(n, 0.5)$$

$$S = \min(n^+, n^-)$$

### Step 4: P-value (Exact)

$$p = 2 \cdot P(X \leq S) = 2 \sum_{k=0}^{S} \binom{n}{k} \left(\frac{1}{2}\right)^n$$

### Step 5: Large-Sample Approximation ($$n > 25$$)

$$z = \frac{n^+ - n/2}{\sqrt{n}/2} = \frac{2n^+ - n}{\sqrt{n}}$$

With continuity correction:

$$z = \frac{|n^+ - n/2| - 0.5}{\sqrt{n}/2}$$

## Comparison: Sign Test vs Wilcoxon Signed-Rank

| Aspect | Sign Test | Wilcoxon Signed-Rank |
|--------|-----------|---------------------|
| Uses magnitudes? | No (only signs) | Yes (ranks of magnitudes) |
| Assumption on differences | None beyond continuity | Symmetry of difference distribution |
| Statistical power | Lower | Higher |
| Handles extreme outliers | Perfectly robust | Somewhat robust |

## Industrial Applications
- **Quality Control**: Proportion of items above/below tolerance (pass/fail)
- **Clinical Trials**: Whether a treatment improves outcomes more often than not
- **Consumer Testing**: Preference tests (which of two products is preferred)
- **Finance**: Whether a portfolio beats the benchmark more often than not

In [0]:
# =============================================================================
# SIGN TEST - Complete Working Example
# =============================================================================

np.random.seed(42)

print("=" * 70)
print("SIGN TEST")
print("=" * 70)

# Scenario: A training program - do more employees improve than decline?
before_training = np.array([65, 72, 58, 80, 74, 68, 77, 63, 70, 85, 60, 73, 67, 79, 71])
after_training = np.array([70, 75, 62, 78, 80, 72, 80, 68, 74, 86, 65, 77, 70, 82, 75])

differences = after_training - before_training
print(f"Differences: {differences}")

# Count signs
n_positive = np.sum(differences > 0)
n_negative = np.sum(differences < 0)
n_zero = np.sum(differences == 0)
n_effective = n_positive + n_negative

print(f"\nn+ (improved): {n_positive}")
print(f"n- (declined): {n_negative}")
print(f"n0 (no change): {n_zero}")
print(f"Effective n: {n_effective}")

# Exact test using binomial distribution
# P-value: probability of observing this extreme or more under H0
S = min(n_positive, n_negative)
p_value_exact = 2 * stats.binom.cdf(S, n_effective, 0.5)

print(f"\n--- Results ---")
print(f"Test statistic S = min(n+, n-) = {S}")
print(f"P-value (exact, binomial): {p_value_exact:.4f}")

# Large-sample Z approximation
z = (2 * n_positive - n_effective) / np.sqrt(n_effective)
z_corrected = (abs(n_positive - n_effective/2) - 0.5) / (np.sqrt(n_effective)/2)
print(f"Z (without correction): {z:.4f}")
print(f"Z (with continuity correction): {z_corrected:.4f}")
print(f"P-value (Z approx): {2 * (1 - stats.norm.cdf(abs(z))):.4f}")

# scipy.stats.binomtest (modern API)
result = stats.binomtest(n_positive, n_effective, p=0.5, alternative='two-sided')
print(f"\nBinomial test p-value: {result.pvalue:.4f}")
print(f"95% CI for proportion of improvements: [{result.proportion_ci().low:.3f}, {result.proportion_ci().high:.3f}]")

# Interpretation
alpha = 0.05
print(f"\n--- Interpretation (α={alpha}) ---")
if p_value_exact < alpha:
    print(f"✓ REJECT H₀: Significantly more employees improved than declined.")
    print(f"  {n_positive}/{n_effective} ({n_positive/n_effective*100:.1f}%) showed improvement.")
else:
    print(f"✗ FAIL TO REJECT H₀: No significant directional trend (p={p_value_exact:.4f})")
    print(f"  {n_positive}/{n_effective} ({n_positive/n_effective*100:.1f}%) improved, but not significantly > 50%.")

print(f"\nNote: The sign test is LESS powerful than Wilcoxon signed-rank because it ignores magnitudes.")
w_stat, w_pval = stats.wilcoxon(before_training, after_training)
print(f"Comparison - Wilcoxon signed-rank p-value: {w_pval:.4f} (uses more information)")

# 13. Runs Test (Wald-Wolfowitz Test)

---

## Overview
The runs test checks whether a sequence of observations is **random** (i.e., the order is not influenced by some pattern or trend). A "run" is a consecutive sequence of similar observations.

## Definition of a Run
Given a binary sequence (e.g., above/below median), a **run** is a maximal consecutive subsequence of identical elements.

Example: $$\texttt{+ + + - - + - - - +}$$ has **5 runs**: $$(+++), (--), (+), (---), (+)$$

## Hypotheses

$$H_0: \text{The sequence is random (observations are independent)}$$

$$H_1: \text{The sequence is not random (shows clustering or mixing)}$$

## Mathematical Formulation

Let:
- $$n_1$$ = number of observations of type 1 (e.g., above median)
- $$n_2$$ = number of observations of type 2 (e.g., below median)
- $$N = n_1 + n_2$$
- $$R$$ = observed number of runs

### Expected Value and Variance Under $$H_0$$

$$E(R) = \frac{2n_1 n_2}{N} + 1$$

$$\text{Var}(R) = \frac{2n_1 n_2 (2n_1 n_2 - N)}{N^2(N-1)}$$

### Test Statistic (Large Sample)

$$z = \frac{R - E(R)}{\sqrt{\text{Var}(R)}}$$

### Decision Rule
- **Too few runs** ($$R$$ is small, $$z < -z_{\alpha/2}$$): evidence of **clustering** (positive autocorrelation)
- **Too many runs** ($$R$$ is large, $$z > z_{\alpha/2}$$): evidence of **mixing/oscillation** (negative autocorrelation)

## Applications in Industry
- **Quality Control**: Testing randomness of defects on a production line (are defects clustered?)
- **Finance**: Testing the random walk hypothesis in stock prices
- **Clinical Trials**: Verifying randomization in treatment assignment sequences
- **Signal Processing**: Detecting patterns in binary communication streams
- **Environmental Monitoring**: Testing for trends in pollution measurements

In [0]:
# =============================================================================
# RUNS TEST (WALD-WOLFOWITZ) - Complete Working Example
# =============================================================================

np.random.seed(42)

print("=" * 70)
print("RUNS TEST FOR RANDOMNESS")
print("=" * 70)

# Scenario: Testing if stock returns show a random pattern
# Simulated daily returns (some with trend, some random)
returns_random = np.random.normal(0.001, 0.02, 50)  # Random
returns_trended = np.concatenate([np.random.normal(0.01, 0.01, 25),  # Uptrend
                                   np.random.normal(-0.01, 0.01, 25)]) # Downtrend

def runs_test(data, cutoff='median'):
    """Perform the runs test for randomness."""
    if cutoff == 'median':
        threshold = np.median(data)
    else:
        threshold = cutoff
    
    # Convert to binary: above/below threshold
    binary = (data > threshold).astype(int)
    
    # Count runs
    runs = 1
    for i in range(1, len(binary)):
        if binary[i] != binary[i-1]:
            runs += 1
    
    n1 = np.sum(binary == 1)  # Above median
    n2 = np.sum(binary == 0)  # Below median
    N = n1 + n2
    
    # Expected value and variance
    E_R = (2 * n1 * n2) / N + 1
    Var_R = (2 * n1 * n2 * (2 * n1 * n2 - N)) / (N**2 * (N - 1))
    
    # Z statistic
    z = (runs - E_R) / np.sqrt(Var_R)
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    
    return {
        'n1': n1, 'n2': n2, 'N': N,
        'runs': runs, 'E_R': E_R, 'Var_R': Var_R,
        'z': z, 'p_value': p_value,
        'binary': binary
    }

# Test random data
print("\n--- Test 1: Random Returns ---")
result_random = runs_test(returns_random)
print(f"n1 (above median): {result_random['n1']}, n2 (below median): {result_random['n2']}")
print(f"Observed runs: {result_random['runs']}")
print(f"Expected runs E(R): {result_random['E_R']:.2f}")
print(f"Std dev: {np.sqrt(result_random['Var_R']):.2f}")
print(f"Z statistic: {result_random['z']:.4f}")
print(f"P-value: {result_random['p_value']:.4f}")
print(f"Conclusion: {'Random (no pattern detected)' if result_random['p_value'] >= 0.05 else 'NOT random'}")

# Test trended data
print("\n--- Test 2: Trended Returns ---")
result_trended = runs_test(returns_trended)
print(f"n1 (above median): {result_trended['n1']}, n2 (below median): {result_trended['n2']}")
print(f"Observed runs: {result_trended['runs']}")
print(f"Expected runs E(R): {result_trended['E_R']:.2f}")
print(f"Z statistic: {result_trended['z']:.4f}")
print(f"P-value: {result_trended['p_value']:.4f}")
print(f"Conclusion: {'Random' if result_trended['p_value'] >= 0.05 else 'NOT random - clustering/trend detected'}")

print("\n--- Interpretation Guide ---")
print("Too FEW runs (z < 0): Clustering/positive autocorrelation (trend)")
print("Too MANY runs (z > 0): Mixing/oscillation (mean-reverting behavior)")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Random sequence
axes[0, 0].plot(returns_random, 'b-', alpha=0.7)
axes[0, 0].axhline(np.median(returns_random), color='red', linestyle='--', label='Median')
axes[0, 0].set_title(f'Random Returns (Runs={result_random["runs"]}, p={result_random["p_value"]:.3f})')
axes[0, 0].legend()

# Random binary pattern
binary_str = ''.join(['■' if b == 1 else '□' for b in result_random['binary']])
colors_r = ['green' if b == 1 else 'red' for b in result_random['binary']]
axes[0, 1].bar(range(len(result_random['binary'])), result_random['binary'] * 2 - 1, 
               color=colors_r, width=1.0)
axes[0, 1].set_title('Binary Pattern (Random) - Green=Above, Red=Below')
axes[0, 1].set_yticks([])

# Trended sequence
axes[1, 0].plot(returns_trended, 'b-', alpha=0.7)
axes[1, 0].axhline(np.median(returns_trended), color='red', linestyle='--', label='Median')
axes[1, 0].set_title(f'Trended Returns (Runs={result_trended["runs"]}, p={result_trended["p_value"]:.3f})')
axes[1, 0].legend()

# Trended binary pattern
colors_t = ['green' if b == 1 else 'red' for b in result_trended['binary']]
axes[1, 1].bar(range(len(result_trended['binary'])), result_trended['binary'] * 2 - 1, 
               color=colors_t, width=1.0)
axes[1, 1].set_title('Binary Pattern (Trended) - Clear Clustering!')
axes[1, 1].set_yticks([])

plt.tight_layout()
plt.show()

# 14. Permutation (Randomization) Tests

---

## Overview
Permutation tests are a class of **exact, non-parametric tests** that derive the sampling distribution by computing the test statistic for **all possible rearrangements** (or a random subset) of the observed data. They are the purest form of non-parametric testing — making virtually no distributional assumptions.

## Core Idea

If $$H_0$$ is true (no group difference), then the group labels are **arbitrary** — any permutation of the labels would be equally likely. We can compute the test statistic for all possible relabelings to build the **null distribution**.

## Mathematical Formulation

### Two-Sample Permutation Test

Given groups $$X = (x_1, \ldots, x_{n_1})$$ and $$Y = (y_1, \ldots, y_{n_2})$$:

1. Choose test statistic: $$T_{obs} = \bar{X} - \bar{Y}$$ (or any relevant statistic)

2. Pool all $$N = n_1 + n_2$$ observations

3. For each possible partition into groups of size $$n_1$$ and $$n_2$$:
   $$T^{(k)} = \bar{X}^{(k)} - \bar{Y}^{(k)} \quad \text{for } k = 1, 2, \ldots, \binom{N}{n_1}$$

4. P-value:
   $$p = \frac{\#\{k : |T^{(k)}| \geq |T_{obs}|\}}{\binom{N}{n_1}}$$

### Monte Carlo Approximation

When $$\binom{N}{n_1}$$ is too large (common in practice), randomly sample $$B$$ permutations:

$$\hat{p} = \frac{1 + \#\{b : |T^{(b)}| \geq |T_{obs}|\}}{B + 1}$$

(The +1 prevents $$p = 0$$ and accounts for the observed data as one permutation.)

## Bootstrap Confidence Interval (Related Technique)

The **bootstrap** resamples **with replacement** to estimate the sampling distribution:

$$\hat{\theta}^{(b)} = T(X_1^*, X_2^*, \ldots, X_n^*) \quad \text{for } b = 1, \ldots, B$$

where $$X_i^*$$ are drawn with replacement from the original sample.

**Percentile CI**: $$[\hat{\theta}^{(\alpha/2)}, \hat{\theta}^{(1-\alpha/2)}]$$

**BCa (Bias-Corrected and Accelerated)**: Adjusts for bias and skewness in the bootstrap distribution.

## Assumptions
1. Observations are **exchangeable** under $$H_0$$ (the only real assumption)
2. For two-sample: groups come from the same distribution under $$H_0$$
3. Independence of observations

## Advantages Over Classical Non-Parametric Tests
- Can test **any** test statistic (mean, variance, ratio, complex functions)
- Exact p-values (not asymptotic approximations)
- No assumptions about distribution shape
- Naturally handle small samples

## Industrial Applications
- **Tech/A/B Testing**: Testing complex metrics (e.g., revenue per user) where CLT may not apply
- **Genomics**: Testing differential gene expression across conditions
- **Clinical Trials**: Small-sample trials where distributional assumptions fail
- **Ecology**: Comparing species diversity between sites
- **Machine Learning**: Permutation feature importance

In [0]:
# =============================================================================
# PERMUTATION TEST & BOOTSTRAP - Complete Working Example
# =============================================================================

np.random.seed(42)

print("=" * 70)
print("PERMUTATION TEST")
print("=" * 70)

# Scenario: Does a new webpage design increase conversion value?
# (Revenue per visitor is heavily right-skewed — parametric tests inappropriate)
control = np.concatenate([np.zeros(80), np.random.exponential(50, 20)])  # 80% zero, 20% purchase
treatment = np.concatenate([np.zeros(70), np.random.exponential(60, 30)])  # 70% zero, 30% purchase

print(f"Control:   n={len(control)}, mean=${control.mean():.2f}, median=${np.median(control):.2f}")
print(f"Treatment: n={len(treatment)}, mean=${treatment.mean():.2f}, median=${np.median(treatment):.2f}")
print(f"Skewness - Control: {stats.skew(control):.2f}, Treatment: {stats.skew(treatment):.2f}")

# Observed test statistic
t_obs = treatment.mean() - control.mean()
print(f"\nObserved difference in means: ${t_obs:.2f}")

# Monte Carlo Permutation Test
B = 10000  # Number of permutations
pooled = np.concatenate([control, treatment])
n1 = len(control)

perm_diffs = np.zeros(B)
for b in range(B):
    perm = np.random.permutation(pooled)
    perm_diffs[b] = perm[n1:].mean() - perm[:n1].mean()

# P-value (two-sided)
p_perm = (1 + np.sum(np.abs(perm_diffs) >= abs(t_obs))) / (B + 1)

print(f"\n--- Permutation Test Results ({B:,} permutations) ---")
print(f"P-value: {p_perm:.4f}")
print(f"Observed statistic is at percentile: {(np.sum(perm_diffs <= t_obs)/B)*100:.1f}%")

# Compare with parametric t-test and Mann-Whitney
t_stat_param, p_param = stats.ttest_ind(treatment, control)
u_stat_mw, p_mw = stats.mannwhitneyu(treatment, control, alternative='two-sided')

print(f"\n--- Comparison with Other Tests ---")
print(f"{'Test':<25} {'P-value':<12} {'Conclusion'}")
print(f"{'Permutation test':<25} {p_perm:<12.4f} {'Significant' if p_perm < 0.05 else 'Not significant'}")
print(f"{'Independent t-test':<25} {p_param:<12.4f} {'Significant' if p_param < 0.05 else 'Not significant'}")
print(f"{'Mann-Whitney U':<25} {p_mw:<12.4f} {'Significant' if p_mw < 0.05 else 'Not significant'}")

# =============================================================================
# BOOTSTRAP CONFIDENCE INTERVAL
# =============================================================================
print("\n" + "=" * 70)
print("BOOTSTRAP CONFIDENCE INTERVAL")
print("=" * 70)

# Bootstrap the difference in means
B_boot = 10000
boot_diffs = np.zeros(B_boot)

for b in range(B_boot):
    boot_control = np.random.choice(control, size=len(control), replace=True)
    boot_treatment = np.random.choice(treatment, size=len(treatment), replace=True)
    boot_diffs[b] = boot_treatment.mean() - boot_control.mean()

# Percentile CI
alpha = 0.05
ci_lower = np.percentile(boot_diffs, 100 * alpha/2)
ci_upper = np.percentile(boot_diffs, 100 * (1 - alpha/2))

print(f"\nBootstrap estimate of mean difference: ${boot_diffs.mean():.2f}")
print(f"Bootstrap SE: ${boot_diffs.std():.2f}")
print(f"95% Percentile CI: [${ci_lower:.2f}, ${ci_upper:.2f}]")
print(f"Does CI contain 0? {'Yes → Not significant' if ci_lower <= 0 <= ci_upper else 'No → Significant'}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Permutation distribution
axes[0].hist(perm_diffs, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(t_obs, color='red', linewidth=2, linestyle='-', label=f'Observed = ${t_obs:.2f}')
axes[0].axvline(-t_obs, color='red', linewidth=2, linestyle='--')
axes[0].set_xlabel('Difference in Means')
axes[0].set_title(f'Permutation Null Distribution\n(p = {p_perm:.4f})')
axes[0].legend()

# Bootstrap distribution
axes[1].hist(boot_diffs, bins=50, density=True, alpha=0.7, color='#2ecc71', edgecolor='black')
axes[1].axvline(ci_lower, color='red', linewidth=2, linestyle='--', label=f'95% CI')
axes[1].axvline(ci_upper, color='red', linewidth=2, linestyle='--')
axes[1].axvline(t_obs, color='black', linewidth=2, label=f'Observed')
axes[1].axvline(0, color='gray', linewidth=1, linestyle=':')
axes[1].set_xlabel('Difference in Means')
axes[1].set_title(f'Bootstrap Distribution\nCI: [${ci_lower:.1f}, ${ci_upper:.1f}]')
axes[1].legend()

# Original data distributions
axes[2].hist(control[control > 0], bins=15, alpha=0.6, label=f'Control (>0)', edgecolor='black', density=True)
axes[2].hist(treatment[treatment > 0], bins=15, alpha=0.6, label=f'Treatment (>0)', edgecolor='black', density=True)
axes[2].set_xlabel('Revenue ($)')
axes[2].set_title('Revenue Distribution (Non-Zero Only)\nHighly Skewed → Permutation Test Ideal')
axes[2].legend()

plt.tight_layout()
plt.show()

# 15. Additional Non-Parametric Tests

---

## 15.1 Anderson-Darling Test

A modification of the KS test with **greater sensitivity to tail deviations**.

$$A^2 = -n - \frac{1}{n} \sum_{i=1}^{n} (2i - 1) \left[ \ln F_0(X_{(i)}) + \ln(1 - F_0(X_{(n+1-i)})) \right]$$

More powerful than KS for detecting departures from normality in the tails.

---

## 15.2 Mann-Kendall Trend Test

Tests for a monotonic trend in time series data. Uses Kendall's $$\tau$$ between time indices and values.

$$S = \sum_{k=1}^{n-1} \sum_{j=k+1}^{n} \text{sgn}(x_j - x_k)$$

$$\text{Var}(S) = \frac{n(n-1)(2n+5)}{18}$$

Applications: Climate trend detection, pollution monitoring, economic indicators.

---

## 15.3 Mood's Median Test

Tests whether $$k$$ groups have the same median. Simpler but less powerful than Kruskal-Wallis.

1. Compute the grand median of all combined observations
2. For each group, count observations above/below the grand median
3. Apply chi-square test to the resulting $$2 \times k$$ contingency table

---

## 15.4 Jonckheere-Terpstra Test

Tests for an **ordered alternative** among $$k$$ groups:

$$H_1: \theta_1 \leq \theta_2 \leq \cdots \leq \theta_k \quad \text{(with at least one strict inequality)}$$

$$J = \sum_{i < j} U_{ij}$$

where $$U_{ij}$$ is the Mann-Whitney U statistic between groups $$i$$ and $$j$$.

More powerful than Kruskal-Wallis when the ordering hypothesis is correct.

---

## 15.5 Cochran's Q Test

Extension of McNemar's test to $$k$$ related binary samples (non-parametric repeated-measures for dichotomous data).

$$Q = \frac{k(k-1) \sum_{j=1}^{k} (C_j - \bar{C})^2}{k \sum_{i=1}^{n} R_i - \sum_{i=1}^{n} R_i^2}$$

where $$C_j$$ = column totals, $$R_i$$ = row totals.

Application: Testing if success rates differ across $$k$$ time points for same subjects.

---

## 15.6 McNemar's Test

For paired $$2 \times 2$$ tables (before/after, matched pairs).

$$\chi^2_{McNemar} = \frac{(b - c)^2}{b + c}$$

where $$b$$ = number changing from 0 to 1, $$c$$ = number changing from 1 to 0.

Application: Testing if a treatment changes the proportion of successes.

In [0]:
# =============================================================================
# ADDITIONAL TESTS - Anderson-Darling, Mann-Kendall, McNemar
# =============================================================================

np.random.seed(42)

print("=" * 70)
print("ANDERSON-DARLING TEST FOR NORMALITY")
print("=" * 70)

# Compare KS vs Anderson-Darling on data with heavy tails
heavy_tails = stats.t.rvs(df=3, size=100)  # t-distribution with 3 df (heavy tails)

ad_result = stats.anderson(heavy_tails, dist='norm')
ks_stat, ks_p = stats.kstest(heavy_tails, 'norm', args=(heavy_tails.mean(), heavy_tails.std()))

print(f"Data: t-distribution with df=3 (heavy-tailed)")
print(f"\nAnderson-Darling: A² = {ad_result.statistic:.4f}")
print(f"Critical values:")
for sl, cv in zip(ad_result.significance_level, ad_result.critical_values):
    reject = "REJECT" if ad_result.statistic > cv else "fail to reject"
    print(f"  α={sl}%: cv={cv:.4f} → {reject}")

print(f"\nKS test: D={ks_stat:.4f}, p={ks_p:.4f}")
print(f"\n→ Anderson-Darling is more sensitive to tail departures from normality")

print("\n" + "=" * 70)
print("MANN-KENDALL TREND TEST")
print("=" * 70)

# Scenario: Is there a trend in monthly temperature anomalies?
months = np.arange(1, 37)  # 3 years of monthly data
temp_anomaly = 0.02 * months + np.random.normal(0, 0.3, 36)  # Slight upward trend + noise

# Compute Mann-Kendall S statistic
n = len(temp_anomaly)
S = 0
for k in range(n-1):
    for j in range(k+1, n):
        S += np.sign(temp_anomaly[j] - temp_anomaly[k])

# Variance of S
var_S = n * (n - 1) * (2*n + 5) / 18

# Z statistic
if S > 0:
    z_mk = (S - 1) / np.sqrt(var_S)
elif S < 0:
    z_mk = (S + 1) / np.sqrt(var_S)
else:
    z_mk = 0

p_mk = 2 * (1 - stats.norm.cdf(abs(z_mk)))

# Kendall's tau
tau_mk = S / (n * (n-1) / 2)

print(f"Monthly temperature anomaly data (n={n})")
print(f"\nMann-Kendall S = {S}")
print(f"Variance(S) = {var_S:.2f}")
print(f"Z = {z_mk:.4f}")
print(f"P-value = {p_mk:.4f}")
print(f"Kendall's τ = {tau_mk:.4f}")
print(f"\nConclusion: {'Significant trend detected' if p_mk < 0.05 else 'No significant trend'}")
print(f"Trend direction: {'Increasing' if S > 0 else 'Decreasing' if S < 0 else 'None'}")

# Sen's slope estimator (median of all pairwise slopes)
slopes = []
for k in range(n-1):
    for j in range(k+1, n):
        slopes.append((temp_anomaly[j] - temp_anomaly[k]) / (j - k))
sen_slope = np.median(slopes)
print(f"Sen's slope: {sen_slope:.4f} per month ({sen_slope*12:.3f} per year)")

print("\n" + "=" * 70)
print("McNEMAR'S TEST")
print("=" * 70)

# Scenario: Did an ad campaign change brand awareness? (same people surveyed before/after)
# Before\After | Aware | Not Aware
# Aware       |   a   |    b
# Not Aware   |   c   |    d

a, b, c, d = 40, 8, 22, 30  # b=lost awareness, c=gained awareness
print(f"\n         After:Aware  After:NotAware")
print(f"Before:Aware     {a}           {b}")
print(f"Before:NotAware  {c}           {d}")
print(f"\nDiscordant pairs: b={b} (lost), c={c} (gained)")

# McNemar's chi-square
chi2_mcnemar = (b - c)**2 / (b + c)
p_mcnemar = 1 - stats.chi2.cdf(chi2_mcnemar, df=1)

# With continuity correction
chi2_corrected = (abs(b - c) - 1)**2 / (b + c)
p_corrected = 1 - stats.chi2.cdf(chi2_corrected, df=1)

# Exact (binomial)
p_exact = 2 * stats.binom.cdf(min(b, c), b + c, 0.5)

print(f"\nχ² (McNemar): {chi2_mcnemar:.4f}, p = {p_mcnemar:.4f}")
print(f"χ² (corrected): {chi2_corrected:.4f}, p = {p_corrected:.4f}")
print(f"Exact (binomial) p-value: {p_exact:.4f}")
print(f"\nConclusion: {'Significant change in awareness' if p_mcnemar < 0.05 else 'No significant change'}")
print(f"Direction: More people GAINED awareness ({c}) than lost it ({b})")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Mann-Kendall trend
axes[0].plot(months, temp_anomaly, 'bo-', markersize=4, alpha=0.7)
axes[0].plot(months, sen_slope * months + np.median(temp_anomaly - sen_slope * months),
            'r-', linewidth=2, label=f"Sen's slope = {sen_slope:.3f}/month")
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Temperature Anomaly (°C)')
axes[0].set_title(f'Mann-Kendall Trend Test (p={p_mk:.3f})')
axes[0].legend()

# McNemar visualization
categories = ['Lost awareness\n(b)', 'Gained awareness\n(c)']
values = [b, c]
colors = ['#e74c3c', '#2ecc71']
axes[1].bar(categories, values, color=colors, edgecolor='black', width=0.5)
axes[1].set_ylabel('Count')
axes[1].set_title(f"McNemar's Test: Discordant Pairs\n(χ²={chi2_mcnemar:.2f}, p={p_mcnemar:.4f})")
for i, v in enumerate(values):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# 16. Comprehensive Decision Guide & Summary

---

## Decision Tree: Choosing the Right Non-Parametric Test

```
┌─ How many groups/samples?
│
├─ ONE SAMPLE
│   ├─ Test location (median)? → Sign Test / Wilcoxon Signed-Rank (vs. hypothesized value)
│   ├─ Test distribution shape? → KS Test / Anderson-Darling / Shapiro-Wilk
│   └─ Test randomness? → Runs Test
│
├─ TWO SAMPLES
│   ├─ Independent?
│   │   ├─ Compare location (median)? → Mann-Whitney U
│   │   ├─ Compare entire distributions? → KS Two-Sample
│   │   └─ Compare proportions (2×2)? → Chi-Square / Fisher's Exact
│   └─ Paired/Related?
│       ├─ Ordinal/continuous data? → Wilcoxon Signed-Rank
│       ├─ Only signs matter? → Sign Test
│       └─ Binary outcomes? → McNemar's Test
│
├─ THREE+ SAMPLES
│   ├─ Independent?
│   │   ├─ No ordering hypothesis? → Kruskal-Wallis H
│   │   ├─ Ordered alternative? → Jonckheere-Terpstra
│   │   └─ Categorical (contingency table)? → Chi-Square Independence
│   └─ Repeated Measures?
│       ├─ Ordinal/continuous? → Friedman Test
│       └─ Binary? → Cochran's Q Test
│
└─ ASSOCIATION/CORRELATION
    ├─ Two ordinal variables? → Spearman ρ / Kendall τ
    ├─ Two categorical variables? → Chi-Square Test of Independence
    └─ Monotonic trend over time? → Mann-Kendall Test
```

---

## Summary Comparison Table

| Test | Parametric Equivalent | Data Type | Purpose | Post-Hoc |
|------|----------------------|-----------|---------|----------|
| Mann-Whitney U | Independent t-test | Ordinal+ | 2 independent groups | N/A |
| Wilcoxon Signed-Rank | Paired t-test | Ordinal+ | 2 paired groups | N/A |
| Sign Test | Paired t-test | Ordinal+ | 2 paired groups (simplest) | N/A |
| Kruskal-Wallis H | One-way ANOVA | Ordinal+ | k independent groups | Dunn's test |
| Friedman | Repeated-measures ANOVA | Ordinal+ | k related groups | Nemenyi / Wilcoxon pairs |
| Chi-Square GOF | Z-test for proportions | Nominal | Observed vs expected frequencies | Standardized residuals |
| Chi-Square Independence | None (unique) | Nominal | Association between categoricals | Standardized residuals |
| KS Test | Normality tests | Continuous | Distribution comparison | N/A |
| Spearman $$\rho$$ | Pearson $$r$$ | Ordinal+ | Monotonic correlation | N/A |
| Kendall $$\tau$$ | Pearson $$r$$ | Ordinal+ | Monotonic correlation | N/A |
| Runs Test | Durbin-Watson | Any | Randomness/independence | N/A |
| Permutation Test | Any parametric test | Any | Universal (any statistic) | Permutation-based |
| Mann-Kendall | Linear regression trend | Ordinal+ | Monotonic trend | N/A |
| McNemar | Paired z-test | Binary | Change in paired proportions | N/A |
| Cochran's Q | Repeated ANOVA (binary) | Binary | k related binary samples | McNemar pairs |

---

## Power Analysis Considerations

The **Asymptotic Relative Efficiency (ARE)** of non-parametric tests relative to their parametric counterparts:

| Test | ARE (vs parametric, normal data) |
|------|----------------------------------|
| Mann-Whitney U vs t-test | $$3/\pi \approx 0.955$$ |
| Wilcoxon Signed-Rank vs paired t | $$3/\pi \approx 0.955$$ |
| Sign Test vs paired t | $$2/\pi \approx 0.637$$ |
| Kruskal-Wallis vs ANOVA | $$3/\pi \approx 0.955$$ |
| Spearman vs Pearson | $$9/\pi^2 \approx 0.912$$ |

Key insight: When data IS normal, non-parametric tests lose only about 5% efficiency. When data is NOT normal, they can be **infinitely more efficient**.

---

## Best Practices

1. **Always visualize first**: Boxplots, histograms, Q-Q plots BEFORE choosing a test
2. **Check assumptions**: Even non-parametric tests have (minimal) assumptions
3. **Report effect sizes**: P-values alone are insufficient; always compute and report effect sizes
4. **Use exact tests for small samples**: Asymptotic approximations fail for $$n < 20$$
5. **Consider permutation tests**: When in doubt, permutation tests are the safest universal option
6. **Multiple comparisons**: Always correct for multiple testing (Bonferroni, Holm, FDR)
7. **Document rationale**: State WHY you chose a non-parametric test (violation of normality, ordinal data, outliers, etc.)

In [0]:
# =============================================================================
# COMPLETE EXAMPLE: Normality Check → Test Selection → Analysis Pipeline
# =============================================================================

np.random.seed(42)

print("=" * 70)
print("FULL ANALYSIS PIPELINE: Choosing Between Parametric & Non-Parametric")
print("=" * 70)

# Scenario: Comparing customer spend across 3 loyalty tiers
# (Real-world spending data is typically right-skewed)
tier_gold = np.random.lognormal(mean=5.5, sigma=0.8, size=40)
tier_silver = np.random.lognormal(mean=5.0, sigma=0.9, size=45)
tier_bronze = np.random.lognormal(mean=4.5, sigma=1.0, size=50)

print("\n" + "-"*40)
print("STEP 1: Descriptive Statistics")
print("-"*40)
for name, data in [('Gold', tier_gold), ('Silver', tier_silver), ('Bronze', tier_bronze)]:
    print(f"{name:8s}: n={len(data):3d}, mean=${data.mean():8.2f}, "
          f"median=${np.median(data):8.2f}, std=${data.std():8.2f}, "
          f"skew={stats.skew(data):.2f}")

print("\n" + "-"*40)
print("STEP 2: Normality Testing")
print("-"*40)
normality_results = []
for name, data in [('Gold', tier_gold), ('Silver', tier_silver), ('Bronze', tier_bronze)]:
    shapiro_stat, shapiro_p = stats.shapiro(data)
    normal = shapiro_p >= 0.05
    normality_results.append(normal)
    print(f"{name:8s}: Shapiro-Wilk W={shapiro_stat:.4f}, p={shapiro_p:.4f} "
          f"{'[✓ Normal]' if normal else '[✗ NOT Normal]'}")

all_normal = all(normality_results)
print(f"\nAll groups normal? {all_normal}")

print("\n" + "-"*40)
print("STEP 3: Homogeneity of Variance")
print("-"*40)
levene_stat, levene_p = stats.levene(tier_gold, tier_silver, tier_bronze)
homogeneous = levene_p >= 0.05
print(f"Levene's test: F={levene_stat:.4f}, p={levene_p:.4f} "
      f"{'[✓ Equal variances]' if homogeneous else '[✗ Unequal variances]'}")

print("\n" + "-"*40)
print("STEP 4: Test Selection")
print("-"*40)
if all_normal and homogeneous:
    print("→ Use: One-way ANOVA (assumptions satisfied)")
    test_used = "ANOVA"
elif all_normal and not homogeneous:
    print("→ Use: Welch's ANOVA (normal but unequal variance)")
    test_used = "Welch ANOVA"
else:
    print("→ Use: Kruskal-Wallis H test (normality violated)")
    print("  Reason: At least one group is not normally distributed")
    test_used = "Kruskal-Wallis"

print("\n" + "-"*40)
print("STEP 5: Hypothesis Testing")
print("-"*40)

# Non-parametric (appropriate choice)
h_stat, kw_p = stats.kruskal(tier_gold, tier_silver, tier_bronze)
print(f"\nKruskal-Wallis H = {h_stat:.4f}, p = {kw_p:.2e}")

# Parametric comparison (for educational purposes)
f_stat, anova_p = stats.f_oneway(tier_gold, tier_silver, tier_bronze)
print(f"ANOVA F = {f_stat:.4f}, p = {anova_p:.2e} (inappropriate but shown for comparison)")

# Effect size
N_total = len(tier_gold) + len(tier_silver) + len(tier_bronze)
epsilon_sq = (h_stat - 3 + 1) / (N_total - 3)
print(f"\nEffect size (ε²): {epsilon_sq:.4f} ({'Small' if epsilon_sq < 0.04 else 'Medium' if epsilon_sq < 0.14 else 'Large'})")

print("\n" + "-"*40)
print("STEP 6: Post-Hoc Analysis")
print("-"*40)
if kw_p < 0.05:
    pairs = [('Gold vs Silver', tier_gold, tier_silver),
             ('Gold vs Bronze', tier_gold, tier_bronze),
             ('Silver vs Bronze', tier_silver, tier_bronze)]
    
    alpha_bonf = 0.05 / 3
    print(f"Bonferroni α = {alpha_bonf:.4f}\n")
    print(f"{'Comparison':<20} {'U':>8} {'p-value':>10} {'Effect (r)':>10} {'Sig':>5}")
    print("-" * 55)
    
    for name, g1, g2 in pairs:
        u, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
        r = 1 - (2*u)/(len(g1)*len(g2))  # rank-biserial
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < alpha_bonf else "ns"
        print(f"{name:<20} {u:>8.0f} {p:>10.4f} {r:>10.3f} {sig:>5}")

print("\n" + "-"*40)
print("STEP 7: Conclusion")
print("-"*40)
print(f"\nTest used: {test_used}")
print(f"Result: {'Significant' if kw_p < 0.05 else 'Not significant'} difference in spending across tiers")
print(f"Practical meaning: Higher loyalty tiers are associated with significantly higher spending.")
print(f"Gold median spend (${np.median(tier_gold):.0f}) > Silver (${np.median(tier_silver):.0f}) > Bronze (${np.median(tier_bronze):.0f})")

# Final visualization
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Box plots
axes[0, 0].boxplot([tier_gold, tier_silver, tier_bronze], 
                   labels=['Gold', 'Silver', 'Bronze'])
axes[0, 0].set_ylabel('Spending ($)')
axes[0, 0].set_title(f'{test_used}: H={h_stat:.2f}, p={kw_p:.2e}')

# Distributions (showing non-normality)
for data, label, color in [(tier_gold, 'Gold', '#FFD700'), 
                            (tier_silver, 'Silver', '#C0C0C0'),
                            (tier_bronze, 'Bronze', '#CD7F32')]:
    axes[0, 1].hist(data, bins=15, alpha=0.5, label=label, color=color, edgecolor='black', density=True)
axes[0, 1].set_xlabel('Spending ($)')
axes[0, 1].set_title('Distributions (Clearly Right-Skewed)')
axes[0, 1].legend()

# Q-Q plot (showing departure from normality)
stats.probplot(tier_bronze, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Bronze Tier) - Departure from Normal')

# Summary of test selection logic
test_info = f"""Test Selection Summary
──────────────────────
1. Normality: {'PASS' if all_normal else 'FAIL'}
2. Homoscedasticity: {'PASS' if homogeneous else 'FAIL'}
3. Decision: {test_used}
4. Result: p = {kw_p:.2e}
5. Effect: ε² = {epsilon_sq:.3f}

Non-parametric test chosen
because normality assumption
is violated (right-skewed data)."""
axes[1, 1].text(0.1, 0.5, test_info, transform=axes[1, 1].transAxes, fontsize=11,
               verticalalignment='center', fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()